In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [2]:
GEN_1_GENETIC = CF_OUTPUTS / "gen1_models_explainers_constraints" / "genetic_exp"
GEN_1_GENETIC.is_dir()

True

In [3]:
batch_data = GEN_1_GENETIC / "RF_prange_highthres_2026-04-29" / "annotated.csv"
batch_df = pd.read_csv(batch_data)

### set PD options

In [4]:
pd.set_option("display.max_rows", None)

In [5]:
# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [6]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

In [7]:
# correct_cf_idx
# Fix cf_id indexing - group by query_index and number counterfactuals sequentially
def fix_cf_id(df):
    # For each query_index group
    fixed_rows = []
    for query_idx, group in df.groupby('query_index', sort=False):
        group = group.copy()
        # First row is xin (original)
        xin_mask = group['cf_id'] == 'xin'
        cf_mask = ~xin_mask

        # Keep xin rows as is
        xin_rows = group[xin_mask].copy()

        # Fix cf rows
        cf_rows = group[cf_mask].copy()
        if len(cf_rows) > 0:
            # Number them cf_1, cf_2, cf_3, etc.
            cf_rows['cf_id'] = [f'cf_{i+1}' for i in range(len(cf_rows))]

        # Combine back
        fixed_rows.append(xin_rows)
        if len(cf_rows) > 0:
            fixed_rows.append(cf_rows)

    return pd.concat(fixed_rows, ignore_index=True)

In [8]:
batch_df = fix_cf_id(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.0,,,,0.0633,
1,0,cf_1,,,,,,,16.7168,,,0.1056,1,False,0.0633,0.0733
2,0,cf_2,,,,,,,16.389,,,0.1208,1,False,0.0633,0.11
3,0,cf_3,,,,7,,,18.9745,,,0.1256,2,False,0.0633,0.07
4,0,cf_4,,,,6,,,18.268,,,0.1168,2,False,0.0633,0.2067
5,0,cf_5,3,,,,,,17.7696,,,0.1191,2,False,0.0633,0.0667
6,0,cf_6,,,,7,,,17.2426,,,0.2061,2,False,0.0633,0.09
7,0,cf_7,,,,,,,16.8547,5,,0.1885,2,False,0.0633,0.1033
8,0,cf_8,3,,,,,,16.389,,,0.1833,2,False,0.0633,0.1267
9,0,cf_9,2,,,,,,16.2995,,,0.25,2,False,0.0633,0.1733


In [9]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation
# prefers valid CFs without violations, and lowest Gower
single_gower_df = select_one_cf_per_query(batch_df, selector="gower")
single_gower_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.0,,,,0.0633,
9,0,cf_10,,,,7,,,18.9745,7,,0.2506,3,True,0.0633,0.03
1,1,xin,3,4,1,2,3,0,22.3757,0,2.1,,,,0.1467,
10,1,cf_9,,,,,1,,22.3757,1,,0.1475,3,True,0.1467,0.05
2,2,xin,5,3,1,1,4,0,22.694,7,2.13,,,,0.1567,
11,2,cf_4,,,,4,2,,22.6757,,,0.1462,3,True,0.1567,0.0633
3,3,xin,3,3,6,6,2,0,24.3418,1,2.21,,,,0.02,
12,3,cf_1,,,,,,,24.3375,,,0.0002,1,True,0.02,0.01
4,4,xin,5,4,2,7,1,0,21.2585,3,2.29,,,,0.0267,
13,4,cf_3,,,3,,,,21.2585,,,0.1431,2,True,0.0267,0.01


In [10]:
# Random selection (Gower may pick only BMI variations)
single_random_df = select_one_cf_per_query(batch_df, selector="random")
single_random_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.0,,,,0.0633,
9,0,cf_10,,,,7,,,18.9745,7,,0.2506,3,True,0.0633,0.03
1,1,xin,3,4,1,2,3,0,22.3757,0,2.1,,,,0.1467,
10,1,cf_5,2,,,,1,,22.3757,,,0.1921,3,True,0.1467,0.01
2,2,xin,5,3,1,1,4,0,22.694,7,2.13,,,,0.1567,
11,2,cf_3,,,,6,1,,22.6757,,,0.2295,3,True,0.1567,0.0633
3,3,xin,3,3,6,6,2,0,24.3418,1,2.21,,,,0.02,
12,3,cf_5,,,,,,,24.3375,3,,0.0502,2,True,0.02,0.0
4,4,xin,5,4,2,7,1,0,21.2585,3,2.29,,,,0.0267,
13,4,cf_3,,,3,,,,21.2585,,,0.1431,2,True,0.0267,0.01
